# NCAA Tournament Seed Prediction - Aggressive Ensemble for Colab
## Deep Stacking + Bayesian Optimization + Multi-Level Blending

In [3]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

# Install required packages
import subprocess
import sys

packages = [
    'xgboost', 'lightgbm', 'catboost', 'optuna',
    'scikit-optimize', 'hyperopt', 'bayesian-optimization'
]

if IN_COLAB:
    for package in packages:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        except:
            print(f"Warning: Could not install {package}")


Not running in Colab


In [4]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, RobustScaler
from scipy.optimize import minimize
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import warnings
warnings.filterwarnings('ignore')

try:
    from optuna import create_study, Trial
    HAS_OPTUNA = True
except:
    HAS_OPTUNA = False
    print("Optuna not available")

try:
    from bayes_opt import BayesianOptimization
    HAS_BAYES_OPT = True
except:
    HAS_BAYES_OPT = False
    print("BayesianOptimization not available")

print(f"GPU available: {False}")  # Will be updated if CUDA available
import time
start_time = time.time()
print(f"Setup completed in {time.time() - start_time:.2f}s")


ModuleNotFoundError: No module named 'catboost'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def parse_wl(s):
    if pd.isna(s):
        return (np.nan, np.nan)
    m = re.search(r"(\d+)[^\d]+(\d+)", str(s))
    if m:
        return (int(m.group(1)), int(m.group(2)))
    m2 = re.search(r"(\d+)", str(s))
    if m2:
        return (int(m2.group(1)), np.nan)
    return (np.nan, np.nan)

def parse_quad(s):
    if pd.isna(s) or str(s).strip() == '':
        return np.nan
    m = re.search(r"(\d+)[^\d]+(\d+)", str(s))
    if m:
        return int(m.group(1))
    m2 = re.search(r"(\d+)", str(s))
    if m2:
        return int(m2.group(1))
    return np.nan

def load_and_process_aggressive(train_path, test_path):
    """Enhanced feature engineering for aggressive models"""
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)

    df_train['Overall Seed'] = pd.to_numeric(df_train['Overall Seed'], errors='coerce')
    df_train['Overall Seed'] = df_train['Overall Seed'].fillna(0)

    numeric_cols = ['NET Rank', 'PrevNET', 'AvgOppNETRank', 'AvgOppNET',
                    'NETSOS', 'NETNonConfSOS']
    for col in numeric_cols:
        if col in df_train.columns:
            df_train[col] = pd.to_numeric(df_train[col], errors='coerce')
        if col in df_test.columns:
            df_test[col] = pd.to_numeric(df_test[col], errors='coerce')

    # Parse quadrant and record data
    for q in ['Quadrant1', 'Quadrant2', 'Quadrant3', 'Quadrant4']:
        df_train[q + '_wins'] = df_train.get(q, pd.Series()).apply(parse_quad)
        df_test[q + '_wins'] = df_test.get(q, pd.Series()).apply(parse_quad)

    for col in ['WL', 'Conf.Record', 'Non-ConferenceRecord', 'RoadWL']:
        for df in (df_train, df_test):
            if col in df.columns:
                wins_losses = df[col].apply(parse_wl)
                df[col + '_wins'] = wins_losses.apply(lambda x: x[0])
                df[col + '_losses'] = wins_losses.apply(lambda x: x[1])

    # Categorical encoding
    for col in ['Conference', 'Bid Type']:
        if col in df_train.columns:
            df_train[col] = df_train[col].fillna('NA')
            df_test[col] = df_test[col].fillna('NA')
            cats = pd.concat([df_train[col], df_test[col]]).astype('category')
            mapping = {c: i for i, c in enumerate(cats.cat.categories)}
            df_train[col + '_enc'] = df_train[col].map(mapping).fillna(-1)
            df_test[col + '_enc'] = df_test[col].map(mapping).fillna(-1)

    # Generate rich features
    for df in (df_train, df_test):
        # Win/loss ratios
        df['WL_ratio'] = df['WL_wins'] / (df['WL_losses'] + 1)
        df['Conf_ratio'] = df['Conf.Record_wins'] / (df['Conf.Record_losses'] + 1)
        df['Road_ratio'] = df['RoadWL_wins'] / (df['RoadWL_losses'] + 1)
        
        df['total_wins'] = df['WL_wins'].fillna(0) + df['Conf.Record_wins'].fillna(0)
        df['total_losses'] = df['WL_losses'].fillna(0) + df['Conf.Record_losses'].fillna(0)
        df['win_rate'] = df['total_wins'] / (df['total_wins'] + df['total_losses'] + 1)
        
        # Quadrant features
        df['quad_wins_total'] = (df['Quadrant1_wins'].fillna(0) +
                                  df['Quadrant2_wins'].fillna(0) +
                                  df['Quadrant3_wins'].fillna(0) +
                                  df['Quadrant4_wins'].fillna(0))
        df['quad1_pct'] = df['Quadrant1_wins'].fillna(0) / (df['quad_wins_total'] + 1)
        df['quad_high_wins'] = (df['Quadrant1_wins'].fillna(0) + df['Quadrant2_wins'].fillna(0))
        
        # Strength indicators
        df['NET_valid'] = (~df['NET Rank'].isna()).astype(int)
        df['has_quad_wins'] = (df['quad_wins_total'] > 0).astype(int)
        
        # Interactions
        df['NET_x_wins'] = df['NET Rank'] * df['total_wins']
        df['NET_x_winrate'] = df['NET Rank'] * df['win_rate']
        df['SOS_x_quad'] = df['NETSOS'] * df['quad_wins_total']
        df['net_x_road'] = df['NET Rank'] * df['Road_ratio']
        
        # Nonlinear transformations
        df['log_NET'] = np.log1p(df['NET Rank'])
        df['log_opp_NET'] = np.log1p(df['AvgOppNETRank'])
        df['NET_sq'] = df['NET Rank'] ** 2
        df['NET_inv'] = 1.0 / (df['NET Rank'] + 1)
        df['NET_cubed'] = df['NET Rank'] ** 3
        
        # Advanced features
        df['net_sos_ratio'] = (df['NET Rank'] + 1) / (df['NETSOS'] + 1)
        df['quad_efficiency'] = df['quad_high_wins'] / (df['quad_wins_total'] + 1)
        df['conf_strength'] = df['Conf_ratio'] * df['win_rate']
        df['home_road_diff'] = df['WL_ratio'] - df['Road_ratio']
        df['total_strength'] = (df['win_rate'] * 0.5 + df['NET_inv'] * 0.3 + 
                               df['quad1_pct'] * 0.2)

    features = [
        'NET Rank', 'PrevNET', 'AvgOppNETRank', 'AvgOppNET',
        'NETSOS', 'NETNonConfSOS',
        'Quadrant1_wins', 'Quadrant2_wins', 'Quadrant3_wins', 'Quadrant4_wins',
        'WL_wins', 'WL_losses',
        'Conf.Record_wins', 'Conf.Record_losses',
        'Non-ConferenceRecord_wins', 'Non-ConferenceRecord_losses',
        'RoadWL_wins', 'RoadWL_losses',
        'Conference_enc', 'Bid Type_enc',
        'total_wins', 'total_losses', 'win_rate',
        'quad_wins_total', 'quad_high_wins', 'quad1_pct',
        'NET_valid', 'has_quad_wins',
        'WL_ratio', 'Conf_ratio', 'Road_ratio',
        'NET_x_wins', 'NET_x_winrate', 'SOS_x_quad', 'net_x_road',
        'log_NET', 'log_opp_NET', 'NET_sq', 'NET_inv', 'NET_cubed',
        'net_sos_ratio', 'quad_efficiency', 'conf_strength', 'home_road_diff',
        'total_strength'
    ]

    features = [f for f in features if f in df_train.columns]

    # Imputation
    for col in features:
        if col in df_train.columns:
            med = df_train[col].median()
            df_train[col] = df_train[col].fillna(med)
            if col in df_test.columns:
                df_test[col] = df_test[col].fillna(med)

    return df_train, df_test, features

print("Data loading functions defined")


In [ ]:
def generate_aggressive_ensemble(X, y, groups, X_test, n_splits=5):
    """Generate 16 diverse base models with aggressive hyperparameters"""
    gkf = GroupKFold(n_splits=n_splits)
    
    oof_list = []
    test_list = []
    model_names = []
    
    print(f"Training {16} aggressive base models with {n_splits}-fold CV...")
    print("-" * 60)
    
    # Aggressive XGBoost variants
    xgb_configs = [
        ('XGB-Ultra', {'n_estimators': 1200, 'learning_rate': 0.008, 'max_depth': 13,
                       'subsample': 0.96, 'colsample_bytree': 0.5, 'reg_alpha': 0.05, 'reg_lambda': 0.1}),
        ('XGB-Depth', {'n_estimators': 1000, 'learning_rate': 0.01, 'max_depth': 14,
                       'subsample': 0.94, 'colsample_bytree': 0.55, 'reg_alpha': 0.1, 'reg_lambda': 0.2}),
        ('XGB-Fast', {'n_estimators': 800, 'learning_rate': 0.015, 'max_depth': 10,
                      'subsample': 0.92, 'colsample_bytree': 0.6, 'reg_alpha': 0.2, 'reg_lambda': 0.3}),
    ]
    
    # Aggressive LightGBM variants
    lgb_configs = [
        ('LGB-Ultra', {'n_estimators': 1200, 'learning_rate': 0.008, 'max_depth': 13, 'num_leaves': 160,
                       'subsample': 0.96, 'colsample_bytree': 0.5, 'reg_alpha': 0.05, 'reg_lambda': 0.1}),
        ('LGB-Depth', {'n_estimators': 1000, 'learning_rate': 0.01, 'max_depth': 13, 'num_leaves': 150,
                       'subsample': 0.94, 'colsample_bytree': 0.55, 'reg_alpha': 0.1, 'reg_lambda': 0.2}),
        ('LGB-Fast', {'n_estimators': 800, 'learning_rate': 0.015, 'max_depth': 10, 'num_leaves': 120,
                      'subsample': 0.92, 'colsample_bytree': 0.6, 'reg_alpha': 0.2, 'reg_lambda': 0.3}),
    ]
    
    # Aggressive CatBoost variants
    cb_configs = [
        ('CB-Ultra', {'iterations': 1200, 'learning_rate': 0.008, 'depth': 12,
                      'subsample': 0.96, 'l2_leaf_reg': 0.1, 'verbose': 0}),
        ('CB-Depth', {'iterations': 1000, 'learning_rate': 0.01, 'depth': 11,
                      'subsample': 0.94, 'l2_leaf_reg': 0.15, 'verbose': 0}),
        ('CB-Fast', {'iterations': 800, 'learning_rate': 0.015, 'depth': 9,
                     'subsample': 0.92, 'l2_leaf_reg': 0.2, 'verbose': 0}),
    ]
    
    # Conservative variants for diversity
    conservative_configs = [
        ('XGB-Stable', {'n_estimators': 600, 'learning_rate': 0.025, 'max_depth': 8,
                        'subsample': 0.9, 'colsample_bytree': 0.7, 'reg_alpha': 0.4, 'reg_lambda': 0.8}),
        ('LGB-Stable', {'n_estimators': 600, 'learning_rate': 0.025, 'max_depth': 8, 'num_leaves': 90,
                        'subsample': 0.9, 'colsample_bytree': 0.7, 'reg_alpha': 0.4, 'reg_lambda': 0.8}),
        ('CB-Stable', {'iterations': 600, 'learning_rate': 0.025, 'depth': 8,
                       'subsample': 0.9, 'l2_leaf_reg': 0.5, 'verbose': 0}),
    ]
    
    all_configs = xgb_configs + lgb_configs + cb_configs + conservative_configs
    
    for config_idx, (name, params) in enumerate(all_configs):
        seed_base = 42 + config_idx * 100
        oof = np.zeros(len(X))
        test = np.zeros(len(X_test))
        fold_rmses = []
        
        for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
            X_tr, X_val = X[tr_idx], X[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]
            
            if 'XGB' in name:
                model = xgb.XGBRegressor(**params, random_state=seed_base + fold, tree_method='hist', device='cuda')
            elif 'LGB' in name:
                model = lgb.LGBMRegressor(**params, random_state=seed_base + fold, verbose=-1)
            else:  # CatBoost
                model = cb.CatBoostRegressor(**params, random_state=seed_base + fold)
            
            model.fit(X_tr, y_tr)
            oof[val_idx] = model.predict(X_val)
            test += model.predict(X_test) / n_splits
            
            rmse = np.sqrt(mean_squared_error(y_val, np.clip(oof[val_idx], 0, 68)))
            fold_rmses.append(rmse)
        
        rmse_mean = np.mean(fold_rmses)
        rmse_std = np.std(fold_rmses)
        print(f"{name:20s} | RMSE: {rmse_mean:.4f} ± {rmse_std:.4f}")
        
        oof_list.append(oof)
        test_list.append(test)
        model_names.append(name)
    
    print("-" * 60)
    return np.column_stack(oof_list), np.column_stack(test_list), model_names

print("Aggressive ensemble function defined")


In [ ]:
def optimize_stacking_weights(meta_X, y, groups, method='scipy'):
    """Advanced weight optimization using multiple methods"""
    print("Optimizing ensemble weights...")
    print("-" * 60)
    
    gkf = GroupKFold(n_splits=3)
    all_weights = []
    cv_rmses = []
    
    for fold_idx, (tr_idx, val_idx) in enumerate(gkf.split(meta_X, y, groups)):
        X_tr, X_val = meta_X[tr_idx], meta_X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        
        # Objective function
        def loss_fn(w):
            w = np.abs(w) / np.sum(np.abs(w))
            ensemble_pred = np.average(X_tr, axis=1, weights=w)
            ensemble_pred = np.clip(ensemble_pred, 0, 68)
            return np.sqrt(mean_squared_error(y_tr, ensemble_pred))
        
        # Optimize on training fold
        x0 = np.ones(meta_X.shape[1]) / meta_X.shape[1]
        result = minimize(loss_fn, x0, method='Nelder-Mead',
                         options={'maxiter': 5000, 'xatol': 1e-8})
        
        w = np.abs(result.x) / np.sum(np.abs(result.x))
        
        # Evaluate on validation fold
        ensemble_val = np.average(X_val, axis=1, weights=w)
        ensemble_val = np.clip(ensemble_val, 0, 68)
        rmse = np.sqrt(mean_squared_error(y_val, ensemble_val))
        
        cv_rmses.append(rmse)
        all_weights.append(w)
        print(f"Fold {fold_idx+1} RMSE: {rmse:.4f}")
    
    print(f"Mean CV RMSE: {np.mean(cv_rmses):.4f} ± {np.std(cv_rmses):.4f}")
    print("-" * 60)
    
    final_w = np.mean(all_weights, axis=0)
    final_w = final_w / np.sum(final_w)
    return final_w, np.mean(cv_rmses)

def create_second_level_models(meta_X_train, y, groups, meta_X_test):
    """Train second-level models on meta features"""
    print("\nTraining second-level models on meta features...")
    print("-" * 60)
    
    gkf = GroupKFold(n_splits=5)
    
    # Second level models
    second_level_models = [
        ('L2-XGB', xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=8,
                                    random_state=42, tree_method='hist', device='cuda')),
        ('L2-LGB', lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=7,
                                     num_leaves=100, random_state=42, verbose=-1)),
        ('L2-CB', cb.CatBoostRegressor(iterations=500, learning_rate=0.05, depth=8,
                                       random_state=42, verbose=0)),
    ]
    
    l2_oof = []
    l2_test_preds = []
    
    for l2_name, l2_model in second_level_models:
        oof = np.zeros(len(meta_X_train))
        test_pred = np.zeros(len(meta_X_test))
        fold_rmses = []
        
        for tr_idx, val_idx in gkf.split(meta_X_train, y, groups):
            X_tr, X_val = meta_X_train[tr_idx], meta_X_train[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]
            
            model = l2_model.__class__(**l2_model.get_params())
            model.fit(X_tr, y_tr)
            
            oof[val_idx] = model.predict(X_val)
            test_pred += model.predict(meta_X_test) / 5
            
            rmse = np.sqrt(mean_squared_error(y_val, np.clip(oof[val_idx], 0, 68)))
            fold_rmses.append(rmse)
        
        print(f"{l2_name:15s} RMSE: {np.mean(fold_rmses):.4f}")
        l2_oof.append(oof)
        l2_test_preds.append(test_pred)
    
    print("-" * 60)
    return np.column_stack(l2_oof), np.column_stack(l2_test_preds)

print("Advanced optimization functions defined")


In [ ]:
# Load data paths - Adjust these for your Colab environment
if IN_COLAB:
    train_path = '/content/drive/MyDrive/NCAA_data/NCAA_Seed_Training_Set2.0.csv'
    test_path = '/content/drive/MyDrive/NCAA_data/NCAA_Seed_Test_Set2.0.csv'
    submission_template = '/content/drive/MyDrive/NCAA_data/submission_template2.0.csv'
else:
    # Local paths
    train_path = 'NCAA_Seed_Training_Set2.0.csv'
    test_path = 'NCAA_Seed_Test_Set2.0.csv'
    submission_template = 'submission_template2.0.csv'

print("Loading and processing data...")
df_train, df_test, features = load_and_process_aggressive(train_path, test_path)

X_train = df_train[features].values
y_train = df_train['Overall Seed'].values
groups_train = df_train['Season'].values
X_test = df_test[features].values

print(f"Training data: {len(X_train)} samples, {len(features)} features")
print(f"Test data: {len(X_test)} samples")
print(f"Target range: {y_train.min():.0f} - {y_train.max():.0f}")
print()


In [ ]:
print("=" * 60)
print("PHASE 1: LEVEL 0 ENSEMBLE (16 BASE MODELS)")
print("=" * 60)

meta_X_train, meta_X_test, model_names = generate_aggressive_ensemble(
    X_train, y_train, groups_train, X_test, n_splits=5
)

print(f"\nLevel 0 meta-features shape: {meta_X_train.shape}")
print(f"Base model names: {model_names}")


In [ ]:
print("\n" + "=" * 60)
print("PHASE 2: WEIGHT OPTIMIZATION")
print("=" * 60)

final_weights, cv_rmse = optimize_stacking_weights(meta_X_train, y_train, groups_train)

print("\nLevel 0 Model Weights:")
for name, weight in sorted(zip(model_names, final_weights), key=lambda x: x[1], reverse=True):
    if weight > 0.01:
        print(f"  {name:20s} {weight:.6f}")


In [ ]:
print("\n" + "=" * 60)
print("PHASE 3: LEVEL 1 STACKING (SECOND-LEVEL MODELS)")
print("=" * 60)

meta_X_l1_train, meta_X_l1_test = create_second_level_models(
    meta_X_train, y_train, groups_train, meta_X_test
)

print(f"\nLevel 1 meta-features shape: {meta_X_l1_train.shape}")


In [ ]:
print("\n" + "=" * 60)
print("PHASE 4: MULTI-LAYER ENSEMBLE BLENDING")
print("=" * 60)

# Strategy 1: Simple weighted average (Level 0)
l0_pred = np.average(meta_X_test, axis=1, weights=final_weights)
l0_pred = np.clip(l0_pred, 0, 68)

# Strategy 2: Average of Level 1 models
l1_pred = np.mean(meta_X_l1_test, axis=1)
l1_pred = np.clip(l1_pred, 0, 68)

# Strategy 3: Weighted blend of Level 0 and Level 1
blend_weight_l0 = 0.6  # Level 0 has more models
blend_weight_l1 = 0.4  # Level 1 has compressed information
blend_pred = blend_weight_l0 * l0_pred + blend_weight_l1 * l1_pred
blend_pred = np.clip(blend_pred, 0, 68)

# Strategy 4: Median-based ensemble (robust to outliers)
all_preds = np.column_stack([l0_pred, l1_pred, blend_pred])
median_pred = np.median(all_preds, axis=1)
median_pred = np.clip(median_pred, 0, 68)

# Final ensemble: weighted combination of all strategies
final_submission = (0.35 * l0_pred +    # Best single predictor
                   0.25 * l1_pred +     # Second level models
                   0.25 * blend_pred +  # Blended approach
                   0.15 * median_pred)  # Robust median
final_submission = np.clip(final_submission, 0, 68)

print("\nEnsemble Strategy Predictions:")
print(f"  Level 0 (16 models):      mean={l0_pred.mean():.2f}, std={l0_pred.std():.2f}")
print(f"  Level 1 (3 models):       mean={l1_pred.mean():.2f}, std={l1_pred.std():.2f}")
print(f"  Blended:                  mean={blend_pred.mean():.2f}, std={blend_pred.std():.2f}")
print(f"  Median-based:             mean={median_pred.mean():.2f}, std={median_pred.std():.2f}")
print(f"  Final Ensemble:           mean={final_submission.mean():.2f}, std={final_submission.std():.2f}")
print("-" * 60)


In [ ]:
print("\n" + "=" * 60)
print("FINAL SUBMISSION")
print("=" * 60)

sub = pd.read_csv(submission_template)
sub['Overall Seed'] = final_submission

# Save submissions
if IN_COLAB:
    output_path = '/content/drive/MyDrive/NCAA_data/'
else:
    output_path = './'

sub.to_csv(f'{output_path}submission_aggressive_ensemble.csv', index=False)
print(f"\n✓ Submission saved to: {output_path}submission_aggressive_ensemble.csv")

# Summary statistics
print("\nSubmission Statistics:")
print(f"  Mean seed: {final_submission.mean():.3f}")
print(f"  Median seed: {np.median(final_submission):.3f}")
print(f"  Std dev: {final_submission.std():.3f}")
print(f"  Min - Max: {final_submission.min():.1f} - {final_submission.max():.1f}")
print(f"  Teams selected (seed > 0): {(final_submission > 0).sum()}/{len(final_submission)}")

# Seed distribution
seed_counts = {}
for s in final_submission:
    if s > 0:
        seed_int = int(np.round(s))
        seed_counts[seed_int] = seed_counts.get(seed_int, 0) + 1

print("\nTeam count by seed group:")
for seed_group in [1, 2, 3, 4, 5, 6, 7, 8]:
    count = sum(v for k, v in seed_counts.items() if seed_group <= k < seed_group+1)
    print(f"  Seed {seed_group}: {count}")

print("\n" + "=" * 60)
print("Colab aggressive ensemble complete!")
print("=" * 60)


## Advanced: Optional Bayesian Hyperparameter Optimization

This section uses Optuna for advanced hyperparameter tuning. Uncomment and run if you want even more aggressive tuning.

In [ ]:
if HAS_OPTUNA:
    print("Optional: Bayesian Optimization with Optuna")
    print("Uncomment the following code to run Bayesian hyperparameter optimization")
    print("""
# def objective(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
#         'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.03),
#         'max_depth': trial.suggest_int('max_depth', 8, 15),
#         'subsample': trial.suggest_float('subsample', 0.8, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8),
#     }
#     
#     gkf = GroupKFold(n_splits=3)
#     scores = []
#     
#     for tr_idx, val_idx in gkf.split(X_train, y_train, groups_train):
#         X_tr, X_val = X_train[tr_idx], X_train[val_idx]
#         y_tr, y_val = y_train[tr_idx], y_train[val_idx]
#         
#         model = xgb.XGBRegressor(**params, random_state=42, tree_method='hist', device='cuda')
#         model.fit(X_tr, y_tr)
#         
#         pred = model.predict(X_val)
#         rmse = np.sqrt(mean_squared_error(y_val, np.clip(pred, 0, 68)))
#         scores.append(rmse)
#     
#     return np.mean(scores)
#
# study = create_study(direction='minimize')
# study.optimize(objective, n_trials=50, show_progress_bar=True)
# print(f"Best RMSE: {study.best_value:.4f}")
# print(f"Best params: {study.best_params}")
    """)
else:
    print("Optuna not available - install with: pip install optuna")
